In [1]:
!mkdir data
!wget https://github.com/mannefedov/compling_nlp_hse_course/raw/master/data/corpus_wsd_50k.txt.zip -P data
!unzip -o data/corpus_wsd_50k.txt.zip -d data/

import numpy as np
import pandas as pd
import time
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet as wn
from sklearn.metrics.pairwise import cosine_distances, cosine_similarity
from sentence_transformers import SentenceTransformer
from sklearn.metrics import accuracy_score
from sklearn.cluster import KMeans, DBSCAN, AffinityPropagation, AgglomerativeClustering, SpectralClustering
from sklearn.metrics import adjusted_rand_score

--2026-10-04 18:47:22--  https://github.com/mannefedov/compling_nlp_hse_course/raw/master/data/corpus_wsd_50k.txt.zip
Resolving github.com (github.com)... 140.82.121.3
Connecting to github.com (github.com)|140.82.121.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/mannefedov/compling_nlp_hse_course/master/data/corpus_wsd_50k.txt.zip [following]
--2026-10-04 18:47:22--  https://raw.githubusercontent.com/mannefedov/compling_nlp_hse_course/master/data/corpus_wsd_50k.txt.zip
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4723095 (4.5M) [application/zip]
Saving to: ‘data/corpus_wsd_50k.txt.zip’

corpus_wsd_50k.txt. 100%[===================>]   4.50M  --.-KB/s    in 0.03s   

2026-10-04 18:47:22 (

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


# Задание 1 (5 балла)

Имплементируйте алгоритм Леска (описание есть в семинаре) и оцените качество его работы на датасете `data/corpus_wsd_50k.txt`

В качестве метрики близости вы должны попробовать два подхода:

1) Jaccard score на множествах слов (определений и контекста)
2) Cosine distance на эмбедингах sentence_transformers

В качестве метрики используйте accuracy (% правильных ответов). Предсказывайте только многозначные слова в датасете

Контекст вы можете определить самостоятельно (окно вокруг целевого слова или все предложение). Также можете поэкспериментировать с предобработкой для обоих методов.

In [2]:
corpus_wsd = []
corpus = open('data/corpus_wsd_50k.txt', encoding='utf-8').read().split('\n\n')
for sent in corpus:
  if sent.strip():
    corpus_wsd.append([s.split('\t') for s in sent.split('\n') if s.strip()])

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_context(words):
    result = []
    for w in words:
        w_lower = w.lower()
        if w_lower.isalpha() and w_lower not in stop_words:
            result.append(lemmatizer.lemmatize(w_lower))
    return result

In [3]:
def jaccard(context_w, synsets):
  scores = []
  context_set = set(context_w)

  for s in synsets:
    definition_set = set(preprocess_context(s.definition().split()))
    intersection = context_set & definition_set
    union = context_set | definition_set

    jaccard_score = len(intersection) / len(union) if union else 0
    scores.append(jaccard_score)

  best_idx = np.argmax(scores)
  return synsets[best_idx]

In [4]:
model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')

def cos_sim(context_embed, synsets):
  definitions = [s.definition() for s in synsets]
  def_embed = model.encode(definitions)

  distances = cosine_distances(context_embed, def_embed)
  best_idx = np.argmin(distances)
  return synsets[best_idx]

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:
y_true = []
y_pred_jaccard = []
y_pred_cosine = []

limit = 500

for sent in corpus_wsd[:limit]:
  words = [token[1] for token in sent if len(token) >= 2 and token[1] != '']
  sentence_text = " ".join(words)
  clean_context = preprocess_context(words)

  context_embed = model.encode(sentence_text).reshape(1, -1)

  for token in sent:
    if '%' in token[0]:
      sense_key = token[0]
      word = token[1]

      try:
        true_synset = wn.lemma_from_key(sense_key).synset()
      except Exception:
        continue
      all_synsets = wn.synsets(word)

      if len(all_synsets) > 1:
        y_true.append(true_synset.name())

        pred_jaccard = jaccard(clean_context, all_synsets)
        y_pred_jaccard.append(pred_jaccard.name())

        pred_cosine = cos_sim(context_embed, all_synsets)
        y_pred_cosine.append(pred_cosine.name())

acc_jaccard = accuracy_score(y_true, y_pred_jaccard)
acc_cosine = accuracy_score(y_true, y_pred_cosine)

print(len(y_true))
print(f"{acc_jaccard * 100:.2f}%")
print(f"{acc_cosine * 100:.2f}%")

3855
33.98%
33.90%


# Задание 2 (5 балла)
Попробуйте разные алгоритмы кластеризации на датасете - `https://github.com/nlpub/russe-wsi-kit/blob/initial/data/main/wiki-wiki/train.csv`

Используйте код из семинара как основу. Используйте ARI как метрику качества.

Попробуйте все 4 алгоритма кластеризации, про которые говорилось на семинаре. Для каждого из алгоритмов попробуйте настраивать гиперпараметры (посмотрите их в документации). Прогоните как минимум 5 экспериментов (не обязательно успешных) с разными параметрами на каждый алгоритме кластеризации и оцените: качество кластеризации, скорость работы, интуитивность параметров.

Помимо этого также выберите 1 дополнительный алгоритм кластеризации отсюда - https://scikit-learn.org/stable/modules/clustering.html , опишите своими словами принцип его работы  и проделайте аналогичные эксперименты.

In [12]:
df = pd.read_csv('https://raw.githubusercontent.com/nlpub/russe-wsi-kit/initial/data/main/wiki-wiki/train.csv', sep='\t')
embedding_model = SentenceTransformer("intfloat/multilingual-e5-base")

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/179k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

...

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

В качестве дополнительного алгоритма был выбран Spectral Clustering, который сначала строит матрицу сходства (граф), где каждый контекст слова связан с соседями на основе близости их эмбеддингов. Затем вычисляются собственные значения и векторы матрицы Лапласа, это позволяет сжать данные и спроецировать их в новое низкоразмерное пространство. На завершающем этапе в этом новом пространстве запускается стандартный алгоритм, который окончательно делит точки на кластеры.

In [16]:
def run_experiment(algo_name, **params):
    start_time = time.time()
    ari_scores = []

    grouped_df = df.groupby('word')

    for word, group in grouped_df:
        contexts = group['context'].tolist()
        true_labels = group['gold_sense_id'].tolist()

        true_k = len(set(true_labels))

        if len(contexts) < 2 or true_k < 2:
            continue

        prefixed_contexts = ["query: " + ctx for ctx in contexts]
        X = embedding_model.encode(prefixed_contexts, show_progress_bar=False)

        if algo_name == 'KMeans':
            model = KMeans(n_clusters=true_k, **params, random_state=42)
        elif algo_name == 'DBSCAN':
            model = DBSCAN(**params)
        elif algo_name == 'Affinity':
            model = AffinityPropagation(**params, random_state=42)
        elif algo_name == 'Agglomerative':
            model = AgglomerativeClustering(n_clusters=true_k, **params)
        elif algo_name == 'Spectral':
            model = SpectralClustering(n_clusters=true_k, **params, random_state=42)

        try:
            preds = model.fit_predict(X)
            ari = adjusted_rand_score(true_labels, preds)
            ari_scores.append(ari)
        except:
            pass

    avg_ari = sum(ari_scores) / len(ari_scores) if ari_scores else 0
    work_time = time.time() - start_time

    print(f"Параметры: {params}")
    print(f"ARI: {avg_ari:.4f}, время: {work_time:.2f} сек\n")

In [17]:
print("KMeans")
run_experiment('KMeans', init='k-means++', n_init=10)
run_experiment('KMeans', init='random', n_init=10)
run_experiment('KMeans', init='k-means++', n_init=1)
run_experiment('KMeans', init='random', n_init=1)
run_experiment('KMeans', algorithm='elkan', n_init=5)

print("\nDBSCAN")
run_experiment('DBSCAN', eps=0.5, min_samples=2)
run_experiment('DBSCAN', eps=0.3, min_samples=3)
run_experiment('DBSCAN', eps=0.8, min_samples=2)
run_experiment('DBSCAN', eps=1.0, min_samples=5)
run_experiment('DBSCAN', eps=0.5, min_samples=5)

print("\nAffinity Propagation")
run_experiment('Affinity', damping=0.5)
run_experiment('Affinity', damping=0.7)
run_experiment('Affinity', damping=0.9)
run_experiment('Affinity', damping=0.6)
run_experiment('Affinity', damping=0.8)

print("\nAgglomerative Clustering")
run_experiment('Agglomerative', linkage='ward')
run_experiment('Agglomerative', linkage='complete')
run_experiment('Agglomerative', linkage='average')
run_experiment('Agglomerative', linkage='single')
run_experiment('Agglomerative', linkage='average', metric='cosine')

print("\nSpectral Clustering")
run_experiment('Spectral', affinity='nearest_neighbors', n_neighbors=5)
run_experiment('Spectral', affinity='nearest_neighbors', n_neighbors=3)
run_experiment('Spectral', affinity='nearest_neighbors', n_neighbors=7)
run_experiment('Spectral', affinity='rbf', gamma=1.0)
run_experiment('Spectral', affinity='rbf', gamma=0.5)

KMeans
Параметры: {'init': 'k-means++', 'n_init': 10}
ARI: 0.6817, время: 3.96 сек

Параметры: {'init': 'random', 'n_init': 10}
ARI: 0.5081, время: 3.95 сек

Параметры: {'init': 'k-means++', 'n_init': 1}
ARI: 0.4071, время: 4.07 сек

Параметры: {'init': 'random', 'n_init': 1}
ARI: 0.4467, время: 4.04 сек

Параметры: {'algorithm': 'elkan', 'n_init': 5}
ARI: 0.5066, время: 4.14 сек


DBSCAN
Параметры: {'eps': 0.5, 'min_samples': 2}
ARI: 0.2298, время: 4.20 сек

Параметры: {'eps': 0.3, 'min_samples': 3}
ARI: -0.0490, время: 4.20 сек

Параметры: {'eps': 0.8, 'min_samples': 2}
ARI: 0.0000, время: 4.26 сек

Параметры: {'eps': 1.0, 'min_samples': 5}
ARI: 0.0000, время: 4.37 сек

Параметры: {'eps': 0.5, 'min_samples': 5}
ARI: 0.2026, время: 4.33 сек


Affinity Propagation
Параметры: {'damping': 0.5}
ARI: 0.0580, время: 4.41 сек

Параметры: {'damping': 0.7}
ARI: 0.0580, время: 4.61 сек

Параметры: {'damping': 0.9}
ARI: 0.0490, время: 4.45 сек

Параметры: {'damping': 0.6}
ARI: 0.0580, время: 4.4

/usr/local/lib/python3.13/dist-packages/sklearn/manifold/_spectral_embedding.py:329: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/manifold/_spectral_embedding.py:329: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/manifold/_spectral_embedding.py:329: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/manifold/_spectral_embedding.py:329: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(


Параметры: {'affinity': 'nearest_neighbors', 'n_neighbors': 3}
ARI: 0.0456, время: 4.14 сек

Параметры: {'affinity': 'nearest_neighbors', 'n_neighbors': 7}
ARI: 0.5909, время: 4.20 сек

Параметры: {'affinity': 'rbf', 'gamma': 1.0}
ARI: 0.5581, время: 4.22 сек

Параметры: {'affinity': 'rbf', 'gamma': 0.5}
ARI: 0.5581, время: 4.20 сек



Наилучшие результаты показали K-Means (с параметрами init=k-means++ и n_init=10, достигнув ARI 0.6817) и Spectral Clustering (с n_neighbors=5, показав ARI 0.6780). Это говорит о том, что они лучше всего работают для задачи WSD.

Для K-Means критически важна качественная инициализация (k-means++) и достаточное количество перезапусков (n_init=10 дает ARI 0.68 против 0.40 при n_init=1). Для Spectral Clustering при выборе слишком малого значения n_neighbors качество резко падает (например, как при n_neighbors=3 до 0.04).

Agglomerative Clustering показал сильный результат при использовании метода связывания ward (ARI 0.5576), так как он минимизирует дисперсию внутри кластеров. Однако другие метрики ARI около нуля или отрицательный.

DBSCAN и Affinity Propagation показали самые слабые результаты. Все алгоритмы показали стабильно высокую скорость работы (в пределах 3.9–4.6 секунд на весь датасет), поскольку векторизация текстов была выполнена заранее, а сами объемы данных сравнительно невелики.